# Phase 1A - TATR table topology (Colab runner)

Runner only: mount, git pull, install, import, call scripts. No logic lives here
(see CLAUDE.md P1/P2). Edit `src/` and `scripts/` locally, push, then re-run.

- **Step 1: inspect** the real FinTabNet.c-Structure layout (already confirmed).
- **Step 2: TATR inference + topology metrics** - run the structure model over a small
  batch, write the prediction/manifest/metrics/failure artifacts, paste the summary back.

## Boot

In [ ]:
# 1. Mount Drive (outputs persist here: manifest / metrics / failures / tables).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Pull the latest code onto the VM (git is the single source of truth).
!cd /content/FinDocStructRAG 2>/dev/null && git pull || git clone https://github.com/AD2000X/FinDocStructRAG.git /content/FinDocStructRAG
%cd /content/FinDocStructRAG

In [ ]:
# 3. Make src/ importable and sanity-check the environment.
import sys
sys.path.insert(0, '/content/FinDocStructRAG')
from src import config, fintabnet_loader
print('IN_COLAB    :', config.IN_COLAB)
print('OUTPUT_ROOT :', config.OUTPUT_ROOT)
print('cache root  :', fintabnet_loader.structure_root())

## Step 1 - inspect the dataset

Downloads + extracts `FinTabNet.c-Structure.tar.gz` to Colab scratch and prints the
layout, file-extension histogram, and a few parsed annotations. Already confirmed; re-run
only if you need to re-check the format.

In [ ]:
# Step 1 needs only huggingface_hub (fast).
!pip install -q huggingface_hub
!python scripts/inspect_fintabnet.py --limit 3

## Step 2 - TATR inference + topology metrics

Installs the GPU stack, then runs the structure model over a small batch. Outputs land
under `outputs/` on Drive (manifest / topology metrics / failures, plus the predicted
tables in `tatr_predicted/`). The run is resumable: re-running skips samples already
marked success in the manifest.

Make sure the runtime is **GPU (T4)**. **Paste back** the final
`processed/skipped/failed` line and the summary JSON (text, not a screenshot).

In [ ]:
# 4. Install the GPU stack for TATR inference (torch/transformers/timm/Pillow/...).
!pip install -q -r requirements-colab.txt

In [ ]:
# 5. Run TATR structure inference + topology metrics over a small smoke batch.
#    Scale up --limit (or set 0 for all) once the loop looks healthy.
!python scripts/run_phase1a_colab.py --limit 10 --run-id smoke